# TD3 baseline on Four Rooms


This notebook implements a **basic baseline RL method** on the same Four Rooms GridWorld environment already used in the project.

Implementation notes:
- Reuses `FourRoomsGridWorld` and `FourRoomsGoalWrapper` from `src/environments/fourrooms.py`.
- Reuses `TrajectoryReplayBuffer` from `src/utils.py` for off-policy methods.
- Keeps action and observation conventions aligned with existing project notebooks.


## Implementation process

1. Use the project environment classes directly (no duplicate environment code).
2. Keep the training loop simple and readable, focusing on a working baseline.
3. Log periodic evaluation returns to monitor learning.
4. Plot return-vs-steps at the end for quick comparison.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

# Resolve repository src path robustly when running notebook in-place.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from environments.fourrooms import FourRoomsGridWorld, FourRoomsGoalWrapper
from utils import TrajectoryReplayBuffer

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
def make_env(goal=(20, 20), slip_prob=0.10, max_horizon=500):
    base = FourRoomsGridWorld(room_size=11, max_episode_steps=max_horizon)
    env = FourRoomsGoalWrapper(
        base,
        goal_position=goal,
        goal_reward=1.0,
        step_reward=0.0,
        slip_prob=slip_prob,
    )
    return env


def evaluate_policy(env, policy_fn, episodes=10):
    returns, lengths = [], []
    for _ in range(episodes):
        obs, _ = env.reset()
        done = False
        ep_ret, ep_len = 0.0, 0
        while not done:
            act = policy_fn(obs)
            obs, rew, term, trunc, _ = env.step(act)
            ep_ret += float(rew)
            ep_len += 1
            done = term or trunc
        returns.append(ep_ret)
        lengths.append(ep_len)
    return float(np.mean(returns)), float(np.mean(lengths))


In [ ]:
# ===== TD3 baseline (deterministic actor, twin critics) =====
class TD3Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, act_dim),
            nn.Tanh(),
        )

    def forward(self, obs):
        return self.net(obs)


class TD3Critic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim + act_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, obs, act):
        return self.net(torch.cat([obs, act], dim=-1))


def collect_episode(env, policy_fn):
    obs, _ = env.reset()
    ep = {k: [] for k in ['obs', 'actions', 'rewards', 'next_obs', 'terminated', 'truncated']}
    done = False
    while not done:
        act = policy_fn(obs)
        next_obs, rew, term, trunc, _ = env.step(act)
        ep['obs'].append(obs.astype(np.float32))
        ep['actions'].append(act.astype(np.float32))
        ep['rewards'].append(np.float32(rew))
        ep['next_obs'].append(next_obs.astype(np.float32))
        ep['terminated'].append(np.float32(term))
        ep['truncated'].append(np.float32(trunc))
        obs = next_obs
        done = term or trunc
    return ep, len(ep['obs'])


def td3_train(
    total_steps=120_000,
    warmup_steps=8_000,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    policy_noise=0.2,
    noise_clip=0.5,
    policy_delay=2,
    expl_noise=0.15,
    lr=3e-4,
):
    env = make_env()
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]

    actor = TD3Actor(obs_dim, act_dim).to(DEVICE)
    actor_tgt = TD3Actor(obs_dim, act_dim).to(DEVICE)
    actor_tgt.load_state_dict(actor.state_dict())

    q1, q2 = TD3Critic(obs_dim, act_dim).to(DEVICE), TD3Critic(obs_dim, act_dim).to(DEVICE)
    q1_tgt, q2_tgt = TD3Critic(obs_dim, act_dim).to(DEVICE), TD3Critic(obs_dim, act_dim).to(DEVICE)
    q1_tgt.load_state_dict(q1.state_dict())
    q2_tgt.load_state_dict(q2.state_dict())

    actor_opt = optim.Adam(actor.parameters(), lr=lr)
    critic_opt = optim.Adam(list(q1.parameters()) + list(q2.parameters()), lr=lr)

    replay = TrajectoryReplayBuffer(capacity=100_000, obs_dim=obs_dim, action_dim=act_dim, device=DEVICE)

    total_env_steps = 0
    train_it = 0
    eval_returns = []

    while total_env_steps < warmup_steps:
        ep, n = collect_episode(env, lambda _: env.action_space.sample())
        replay.add_episode(ep)
        total_env_steps += n

    while total_env_steps < total_steps:
        def behavior(obs):
            obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                a = actor(obs_t).squeeze(0).cpu().numpy()
            a = a + np.random.normal(0.0, expl_noise, size=act_dim)
            return np.clip(a, -1.0, 1.0).astype(np.float32)

        ep, n = collect_episode(env, behavior)
        replay.add_episode(ep)
        total_env_steps += n

        for _ in range(max(1, n // 2)):
            train_it += 1
            batch = replay.sample(batch_size)
            obs = batch.obs.float()
            act = batch.actions.float()
            rew = batch.rewards.float()
            next_obs = batch.next_obs.float()
            done = torch.clamp(batch.terminated + batch.truncated, 0, 1).float()

            with torch.no_grad():
                noise = torch.randn_like(act) * policy_noise
                noise = torch.clamp(noise, -noise_clip, noise_clip)
                next_a = torch.clamp(actor_tgt(next_obs) + noise, -1.0, 1.0)
                target_q = torch.min(q1_tgt(next_obs, next_a), q2_tgt(next_obs, next_a))
                y = rew + gamma * (1 - done) * target_q

            critic_loss = F.mse_loss(q1(obs, act), y) + F.mse_loss(q2(obs, act), y)
            critic_opt.zero_grad()
            critic_loss.backward()
            critic_opt.step()

            if train_it % policy_delay == 0:
                actor_loss = -q1(obs, actor(obs)).mean()
                actor_opt.zero_grad()
                actor_loss.backward()
                actor_opt.step()

                with torch.no_grad():
                    for p, pt in zip(actor.parameters(), actor_tgt.parameters()):
                        pt.data.mul_(1 - tau).add_(tau * p.data)
                    for p, pt in zip(q1.parameters(), q1_tgt.parameters()):
                        pt.data.mul_(1 - tau).add_(tau * p.data)
                    for p, pt in zip(q2.parameters(), q2_tgt.parameters()):
                        pt.data.mul_(1 - tau).add_(tau * p.data)

        if train_it % 300 == 0:
            eval_env = make_env()
            mean_ret, mean_len = evaluate_policy(
                eval_env,
                lambda o: actor(torch.tensor(o, dtype=torch.float32, device=DEVICE).unsqueeze(0))
                .squeeze(0).detach().cpu().numpy().astype(np.float32),
                episodes=8,
            )
            eval_returns.append((total_env_steps, mean_ret))
            print(f'[TD3] step={total_env_steps:7d} | eval_return={mean_ret:.3f} | eval_len={mean_len:.1f}')
            eval_env.close()

    env.close()
    return actor, eval_returns


td3_actor, td3_eval = td3_train()

if td3_eval:
    xs, ys = zip(*td3_eval)
    plt.figure(figsize=(7, 4))
    plt.plot(xs, ys)
    plt.xlabel('Environment steps')
    plt.ylabel('Mean episodic return')
    plt.title('TD3 baseline on FourRooms')
    plt.grid(alpha=0.25)
    plt.show()
